# Batching Example

This notebook demonstrates how to create a batch request and track its status with the GigaChat resource API.


Import required libraries for working with batch files and the GigaChat client.


In [ ]:
import base64
import getpass
import os

from gigachat import GigaChat


Set up GigaChat credentials and scope. The input will be hidden for security.


In [ ]:
if "GIGACHAT_CREDENTIALS" not in os.environ:
    os.environ["GIGACHAT_CREDENTIALS"] = getpass.getpass("GigaChat Credentials: ")

if "GIGACHAT_SCOPE" not in os.environ:
    os.environ["GIGACHAT_SCOPE"] = getpass.getpass("GigaChat Scope: ")


## Load Batch Input File

Read a JSONL file with batch requests. The API currently supports up to 500 lines per file.


In [ ]:
with open("chat_batch_100.jsonl", "rb") as f:
    data = f.read()


## Create Batch

Create a batch using the appropriate method for your file, such as `chat_completions` or `embedder`.


In [ ]:
with GigaChat(verify_ssl_certs=False) as giga:
    batch_creation = giga.batches.create(data, method="chat_completions")
    print(batch_creation)


## Check Batch Status

Batch jobs move through `created`, `in_progress`, and `completed` states. Use `batches.retrieve` to poll one batch, or `batches.list` to inspect recent batch jobs.


In [ ]:
with GigaChat(verify_ssl_certs=False) as giga:
    result = giga.batches.retrieve(batch_creation.id_)
    print(result)


In [ ]:
with GigaChat(verify_ssl_certs=False) as giga:
    batches = giga.batches.list()
    print(batches)


## Download Output

Use `giga.files.retrieve_content(result.batches[0].output_file_id)` to fetch the output file and save the decoded JSONL payload.


In [ ]:
output_file_id = result.batches[0].output_file_id
if output_file_id is None:
    print("Batch output is not available yet")
else:
    with GigaChat(verify_ssl_certs=False) as giga:
        output = giga.files.retrieve_content(output_file_id)

    with open("batch_output.jsonl", "wb") as f:
        f.write(base64.b64decode(output.content))
